In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import lu_factor, lu_solve, lu

# Load the dataset
df = pd.read_csv('dataset/ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [2]:
# Sparse matrix
R = df.pivot(index='user_id', columns='item_id', values='rating').to_numpy()
R = np.nan_to_num(R) 
R

array([[5., 3., 4., ..., 0., 0., 0.],
       [4., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 5., 0., ..., 0., 0., 0.]], shape=(943, 1682))

In [3]:
# Train-test split
def train_test_split(R, test_ratio=0.2, seed=42):
    np.random.seed(seed)
    R_train, R_test = R.copy(), np.zeros_like(R)
    observed = list(zip(*np.where(R > 0)))
    np.random.shuffle(observed)
    for u, i in observed[:int(len(observed) * test_ratio)]:
        R_test[u, i] = R[u, i]
        R_train[u, i] = 0.0
    return R_train, R_test

In [4]:
# Summary
print(f"Users         : {R.shape[0]}")
print(f"Items         : {R.shape[1]}")
print(f"Ratings       : {np.count_nonzero(R)}")
print(f"Sparsity      : {100 * (1 - np.count_nonzero(R) / R.size):.2f}%")
R_train, R_test = train_test_split(R)
print(f"Train ratings : {np.count_nonzero(R_train)}")
print(f"Test ratings  : {np.count_nonzero(R_test)}")


Users         : 943
Items         : 1682
Ratings       : 100000
Sparsity      : 93.70%
Train ratings : 80000
Test ratings  : 20000


System of Equation Solver

In [5]:
# Print the SPD system of equations
np.random.seed(7)
k = 5
M = np.random.randn(k,k)
A = M.T @ M +  0.5 * np.eye(k)  # Make it SPD
b = np.random.randn(k)
print(f"A:\n{np.round(A, 2)}")
print(f"b:\n{np.round(b, 2)}")

x_true = np.linalg.solve(A, b)
print(f"True solution:\n{np.round(x_true, 2)}")




A:
[[ 8.59 -1.23 -1.08  3.81  0.96]
 [-1.23  1.08 -0.09  0.32 -0.44]
 [-1.08 -0.09  4.   -2.66 -1.37]
 [ 3.81  0.32 -2.66  5.96 -0.16]
 [ 0.96 -0.44 -1.37 -0.16  3.87]]
b:
[-1.45 -0.41 -2.29  1.05 -0.42]
True solution:
[-0.5  -1.22 -0.7   0.24 -0.36]


In [6]:
# Matrix Invserse Solver
def solve_inverse(A, b):
    A_inv = np.linalg.inv(A)
    x = A_inv @ b
    res = np.linalg.norm(A @ x - b)
    return x, res

x_inv, res_inv = solve_inverse(A, b)
print(f"Inverse solution:\n{np.round(x_inv, 2)}")
print(f"Residual norm: {res_inv:.2e}")


Inverse solution:
[-0.5  -1.22 -0.7   0.24 -0.36]
Residual norm: 5.12e-16


In [7]:
# Gaussian Elimination Solver
def solve_gaussian_elimination(A, b):
    k = len(b)
    M = np.hstack([A.astype(float).copy(), 
                    b.reshape(-1, 1).astype(float)])
    
    for col in range(k):
        # Partial pivoting
        max_row = col + np.argmax(np.abs(M[col:, col]))
        M[[col, max_row]] = M[[max_row, col]]

        pivot = M[col, col]
        for row in range(col + 1, k):
            factor = M[row, col] / pivot
            M[row,col:] -= factor * M[col, col:]

      # Back substitution
    x = np.zeros(k)
    for i in range(k - 1, -1, -1):
        x[i] = (M[i,k] - np.dot(M[i, i + 1:k], x[i + 1:k])) / M[i, i]
        res = float(np.linalg.norm(A @ x - b))
    return x, res

x_gauss, res_gauss = solve_gaussian_elimination(A, b)
print(f"Gaussian elimination solution:\n{np.round(x_gauss, 2)}")
print(f"Residual norm: {res_gauss:.2e}")

Gaussian elimination solution:
[-0.5  -1.22 -0.7   0.24 -0.36]
Residual norm: 5.55e-17


In [8]:
# LU Decomposition Solver
def solve_lu(A, b):
    P, L, U = lu(A)

    lu_piv = lu_factor(A)
    x = lu_solve(lu_piv, b)
    res = float(np.linalg.norm(A @ x - b))
    return x, res, L, U

x_lu, res_lu, L, U = solve_lu(A, b)
print(f"L:\n{np.round(L, 2)}")
print(f"U:\n{np.round(U, 2)}")
print(f"LU decomposition solution:\n{np.round(x_lu, 2)}")
print(f"Residual norm: {res_lu:.2e}")

L:
[[ 1.    0.    0.    0.    0.  ]
 [-0.14  1.    0.    0.    0.  ]
 [-0.13 -0.27  1.    0.    0.  ]
 [ 0.44  0.96 -0.51  1.    0.  ]
 [ 0.11 -0.34 -0.35 -0.4   1.  ]]
U:
[[ 8.59 -1.23 -1.08  3.81  0.96]
 [ 0.    0.9  -0.25  0.87 -0.31]
 [ 0.    0.    3.8  -1.94 -1.34]
 [ 0.    0.    0.    2.46 -0.97]
 [ 0.    0.    0.    0.    2.81]]
LU decomposition solution:
[-0.5  -1.22 -0.7   0.24 -0.36]
Residual norm: 6.13e-16


In [9]:
# QR Decomposition Solver
def solve_qr(A, b):
    Q, R = np.linalg.qr(A)
    b_hat = Q.T @ b
    k = len(b)
    x = np.zeros(k)
    for i in range(k - 1, -1, -1):
        x[i] = (b_hat[i] - np.dot(R[i, i + 1:], x[i + 1:])) / R[i, i]
    res = float(np.linalg.norm(A @ x - b))
    return x, res, Q, R

x_qr, res_qr, Q, R_mat = solve_qr(A, b)
print(f"Q:\n{np.round(Q, 2)}")
print(f"R:\n{np.round(R_mat, 2)}")
print(f"QR decomposition solution:\n{np.round(x_qr, 2)}")
print(f"Residual norm: {res_qr:.2e}")

Q:
[[-0.9   0.16 -0.24  0.3  -0.16]
 [ 0.13 -0.72 -0.22  0.64  0.09]
 [ 0.11  0.17 -0.85 -0.19  0.45]
 [-0.4  -0.6   0.13 -0.6   0.32]
 [-0.1   0.26  0.39  0.33  0.81]]
R:
[[-9.58  1.15  2.6  -6.02 -1.4 ]
 [ 0.   -1.29  1.83 -3.72  1.32]
 [ 0.    0.   -4.    1.98  2.54]
 [ 0.    0.    0.   -1.8   1.64]
 [ 0.    0.    0.    0.    2.28]]
QR decomposition solution:
[-0.5  -1.22 -0.7   0.24 -0.36]
Residual norm: 1.31e-15


In [10]:
# Gauss-Jacobi Iteration Solver
def solve_gauss_jacobi(A, b, max_iter=1000, tol=1e-6, track=True):
    k = len(b)
    D_inv = 1.0 / np.diag(A)
    R_off = A - np.diag(np.diag(A))
    x = np.zeros(k)
    history = []

    for it in range(1, max_iter + 1):
        x_new = D_inv * (b - R_off @ x)
        err = np.linalg.norm(x_new -x)
        history.append(float(np.linalg.norm(A @ x_new - b)))
        if err < tol:
            x = x_new
            break
        x = x_new
      
    res = float(np.linalg.norm(A @ x - b))
    if track:
        print(f'Converged in {it} iterations')
        for i in range(0, len(history), max(1, len(history) // 10)):
            print(f'Iter {i:4d}: Residual norm = {history[i]:.2e}')
    return x, res, history

x_gj, res_gj, history_gj = solve_gauss_jacobi(A, b)
print(f"Gauss-Jacobi solution:\n{np.round(x_gj, 2)}")
print(f"Residual norm: {res_gj:.2e}")

Converged in 351 iterations
Iter    0: Residual norm = 2.01e+00
Iter   35: Residual norm = 3.83e-01
Iter   70: Residual norm = 1.10e-01
Iter  105: Residual norm = 3.15e-02
Iter  140: Residual norm = 9.04e-03
Iter  175: Residual norm = 2.59e-03
Iter  210: Residual norm = 7.43e-04
Iter  245: Residual norm = 2.13e-04
Iter  280: Residual norm = 6.11e-05
Iter  315: Residual norm = 1.75e-05
Iter  350: Residual norm = 5.02e-06
Gauss-Jacobi solution:
[-0.5  -1.22 -0.7   0.24 -0.36]
Residual norm: 5.02e-06


In [11]:
# Gauss-Seidel Iterative Solver
def solve_gauss_seidel(A, b, max_iter=1000, tol=1e-6, track=True):
    k = len(b)
    x = np.zeros(k)
    history = []

    for it in range(1, max_iter + 1):
        x_old = x.copy()
        for i in range(k):
            sum1 = np.dot(A[i, :i], x[:i])
            sum2 = np.dot(A[i, i + 1:], x_old[i + 1:])
            x[i] = (b[i] - sum1 - sum2) / A[i, i]
        err = np.linalg.norm(x - x_old)
        history.append(float(np.linalg.norm(A @ x - b)))
        if err < tol:
            break

    res = float(np.linalg.norm(A @ x - b))
    if track:
        print(f'Converged in {it} iterations')
        for i in range(0, len(history), max(1, len(history) // 10)):
            print(f'Iter {i:4d}: Residual norm = {history[i]:.2e}')
    return x, res, history

x_gs, res_gs, history_gs = solve_gauss_seidel(A, b)
print(f"Gauss-Seidel solution:\n{np.round(x_gs, 2)}")
print(f"Residual norm: {res_gs:.2e}")

Converged in 32 iterations
Iter    0: Residual norm = 1.26e+00
Iter    3: Residual norm = 2.45e-01
Iter    6: Residual norm = 7.09e-02
Iter    9: Residual norm = 2.06e-02
Iter   12: Residual norm = 6.00e-03
Iter   15: Residual norm = 1.74e-03
Iter   18: Residual norm = 5.07e-04
Iter   21: Residual norm = 1.47e-04
Iter   24: Residual norm = 4.28e-05
Iter   27: Residual norm = 1.24e-05
Iter   30: Residual norm = 3.61e-06
Gauss-Seidel solution:
[-0.5  -1.22 -0.7   0.24 -0.36]
Residual norm: 2.39e-06


Alternating Least Squares (ALS) is a method used to solve systems of equations, particularly in the context of matrix factorization and collaborative filtering. The ALS algorithm iteratively optimizes one set of variables while keeping the other set fixed, alternating between them until convergence.

In [12]:
n_users, n_items = R.shape
k = 10  # Number of latent factors
lam = 0.1  # Regularization parameter

np.random.seed(42)
U = np.random.normal(0, 0.1, (n_users, k))
V = np.random.normal(0, 0.1, (n_items, k))


print(f"U shape: {U.shape}")
print(f"V shape: {V.shape}")
print()
print(f"U[0]: {np.round(U[0], 4)}")
print(f"V[0]: {np.round(V[0], 4)}")
print()
print(f'U norm: {np.linalg.norm(U):.4f}')
print(f'V norm: {np.linalg.norm(V):.4f}')

U shape: (943, 10)
V shape: (1682, 10)

U[0]: [ 0.0497 -0.0138  0.0648  0.1523 -0.0234 -0.0234  0.1579  0.0767 -0.0469
  0.0543]
V[0]: [ 0.0863 -0.083  -0.0282 -0.0637  0.1543 -0.142   0.0114 -0.0109  0.0827
 -0.0104]

U norm: 9.7651
V norm: 12.8902
